<a href="https://colab.research.google.com/github/sebastianatanasovici-wq/-notebook-/blob/main/pregunta_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.8 MB/s eta 0:00:00


# **6A)Tablones**

***Restricciones críticas:***

Demanda total de madera

Tiempo de secado

Capacidad del aserradero

***Interpretación:***

Se priorizan las fuentes más baratas y eficientes

El tiempo de secado puede ser el cuello de botella

***Sensibilidad:***

Si aumenta el tiempo disponible → baja el costo total

Si sube el costo de compra → se usan más troncos

Si aumenta la demanda → aumenta el costo total

***Variables***

x1: tablones grado 1 comprados

x2: tablones grado 2 comprados

x3: troncos procesados

***Rendimientos***

Grado 1 → 0.7

Grado 2 → 0.9

Troncos → 0.8
Costos

Compra:

grado 1: 3

grado 2: 7

Troncos:

cortar: 3

aserradero: 2.5

Secado:

4 por unidad


In [9]:
def create_ampl_instance(solver: str = 'highs'):
    return ampl_notebook(modules=[solver], license_uuid='default')


def extract_solution(ampl):
    return {
        "Grado1": ampl.get_variable("x1").value(),
        "Grado2": ampl.get_variable("x2").value(),
        "Troncos": ampl.get_variable("x3").value(),
        "Costo": ampl.get_objective("Costo").value(),
    }


@timed
def solve_brady():
    ampl = create_ampl_instance()

    ampl.eval(r'''
        var x1 >= 0;
        var x2 >= 0;
        var x3 >= 0;

        minimize Costo:
            3*x1 + 7*x2
            + (3+2.5)*x3
            + 4*(0.7*x1 + 0.9*x2 + 0.8*x3);

        r1: 0.7*x1 + 0.9*x2 + 0.8*x3 >= 90000;
        r2: x1 <= 40000;
        r3: x2 <= 60000;
        r4: x3 <= 35000;
        r5: 1.2*x1 + 0.8*x2 + 1.3*x3 <= 144000;
    ''')

    ampl.option["solver"] = "highs"
    ampl.solve()

    return extract_solution(ampl)

resultado, tiempo = solve_brady()

print("=== RESULTADOS EJERCICIO 6A (BRADY) ===")
print(f"Solución -> {resultado}")
print(f"Tiempo   -> {tiempo:.6f} segundos")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 936944.4444
0 simplex iterations
0 barrier iterations
=== RESULTADOS EJERCICIO 6A (BRADY) ===
Solución -> {'Grado1': 40000.0, 'Grado2': 37777.77777777777, 'Troncos': 35000.0, 'Costo': 936944.4444444443}
Tiempo   -> 10.325672 segundos


# **6B) Chatarras**

***Restricciones activas:***

Composición química

***Interpretación:***

La mezcla óptima depende de cumplir límites de calidad al menor costo

***Sensibilidad:***

Si baja el costo de A → se usa más A

Si cambian los límites de composición → cambia la mezcla

Si aumenta la cantidad total (1000) → el problema escala proporcionalmente

***Variables***

x: toneladas de chatarra A

y: toneladas de chatarra B

***Costos***

A: 100
B: 80

In [10]:
def extract_solution_chatarras(ampl):
    return {
        "Chatarra_A": ampl.get_variable("x").value(),
        "Chatarra_B": ampl.get_variable("y").value(),
        "Costo": ampl.get_objective("Costo").value(),
    }


@timed
def solve_chatarras():
    ampl = create_ampl_instance()

    ampl.eval(r'''
        var x >= 0;
        var y >= 0;

        minimize Costo:
            100*x + 80*y;

        r1: x + y = 1000;

        # Aluminio
        r2: 0.06*x + 0.03*y >= 0.03*1000;
        r3: 0.06*x + 0.03*y <= 0.06*1000;

        # Silicio
        r4: 0.03*x + 0.06*y >= 0.03*1000;
        r5: 0.03*x + 0.06*y <= 0.05*1000;

        # Carbono
        r6: 0.04*x + 0.03*y >= 0.03*1000;
        r7: 0.04*x + 0.03*y <= 0.07*1000;
    ''')

    ampl.option["solver"] = "highs"
    ampl.solve()

    return extract_solution_chatarras(ampl)

resultado2, tiempo2 = solve_chatarras()

print("=== RESULTADOS EJERCICIO 6B (CHATARRAS) ===")
print(f"Solución -> {resultado2}")
print(f"Tiempo   -> {tiempo2:.6f} segundos")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 86666.66667
0 simplex iterations
0 barrier iterations
=== RESULTADOS EJERCICIO 6B (CHATARRAS) ===
Solución -> {'Chatarra_A': 333.3333333333334, 'Chatarra_B': 666.6666666666666, 'Costo': 86666.66666666667}
Tiempo   -> 5.039577 segundos
